# 03D. Regime-Conditioned IC Engine

Measure where each candidate signal works or breaks by splitting cross-sectional IC performance across market regimes. This notebook is diagnostic only and does not approve/reject signals, replace WFV, or modify scoring, WFV, decay, composite, alpha, stress/freeze, portfolio, or ML logic.

## 1. Purpose and scope

Use existing candidate signal and scoring artifacts to compute regime-conditioned IC summaries and fragility diagnostics. High regime fragility is conditionality evidence, not an automatic rejection.

## 2. Imports and config

In [1]:
from __future__ import annotations

import gc
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.db import get_db_path, load_price_table, load_table
from src.regime_ic import (
    build_regime_features_for_ic,
    build_regime_opportunity_summary,
    run_regime_ic_analysis,
)
from src.regime_ic_storage import REGIME_IC_TABLES, save_regime_ic_outputs
from src.run_config import make_run_id, make_run_timestamp
from src.signal_storage import (
    load_candidate_signals_by_names,
    validate_signal_date_quality,
    validate_signal_long_uniqueness,
)

REGIME_IC_VERSION = 'phase2_regime_ic_v1'
REGIME_COLUMNS = [
    'benchmark_vol_regime',
    'benchmark_trend_regime',
    'drawdown_regime',
    'correlation_regime',
]
HORIZONS = [1, 5, 10, 20]
IC_METHOD = 'spearman'

sqlite_db_path = get_db_path()

print(f'SQLite database: {sqlite_db_path}')
print(f'Regime IC version: {REGIME_IC_VERSION}')

SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Regime IC version: phase2_regime_ic_v1


## 3. Create regime IC run_id / timestamp

In [2]:
run_id = make_run_id(prefix='phase2_nb03d_regime_ic')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')

run_id: phase2_nb03d_regime_ic_20260510_215329
run_timestamp: 2026-05-10 21:53:29


## 4. Load candidate_signals_current

In [3]:
signal_scores = load_table('signal_scores_current', db_path=sqlite_db_path)
signal_best_horizon = load_table('signal_best_horizon_current', db_path=sqlite_db_path)

needed_signal_names = (
    signal_scores.loc[signal_scores['horizon'].isin(HORIZONS), 'signal_name']
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print('needed_signal_names:', needed_signal_names)
print('count:', len(needed_signal_names))

with sqlite3.connect(sqlite_db_path) as conn:
    sqlite_date_probe_rows = []
    for name in needed_signal_names:
        summary = pd.read_sql_query(
            """
            SELECT
                COUNT(*) AS rows,
                MIN(Date) AS min_date,
                MAX(Date) AS max_date,
                SUM(CASE WHEN Date IS NULL THEN 1 ELSE 0 END) AS date_nulls
            FROM candidate_signals_current
            WHERE signal_name = ?
            """,
            conn,
            params=[name],
        ).iloc[0].to_dict()
        sample_dates = pd.read_sql_query(
            """
            SELECT Date
            FROM candidate_signals_current
            WHERE signal_name = ?
            LIMIT 5
            """,
            conn,
            params=[name],
        )['Date'].tolist()
        sqlite_date_probe_rows.append(
            {
                'signal_name': name,
                'rows': int(summary['rows']),
                'min_date': summary['min_date'],
                'max_date': summary['max_date'],
                'date_nulls': int(summary['date_nulls'] or 0),
                'sample_dates': sample_dates,
            }
        )

sqlite_date_probe = pd.DataFrame(sqlite_date_probe_rows)
display(sqlite_date_probe)

single_signal_date_probe_rows = []
for name in needed_signal_names:
    df_one = load_candidate_signals_by_names(
        [name],
        current=True,
        db_path=sqlite_db_path,
        chunksize=500_000,
    )
    one_diag = {
        'signal_name': name,
        'rows': len(df_one),
        'Date_dtype': str(df_one['Date'].dtype),
        'date_nulls': int(df_one['Date'].isna().sum()),
        'unique_dates': int(df_one['Date'].nunique()),
    }
    single_signal_date_probe_rows.append(one_diag)
    print(name, len(df_one), df_one['Date'].dtype, df_one['Date'].isna().sum(), df_one['Date'].nunique())
    del df_one
    gc.collect()

single_signal_date_probe = pd.DataFrame(single_signal_date_probe_rows)
display(single_signal_date_probe.sort_values('date_nulls', ascending=False))

candidate_signals = load_candidate_signals_by_names(
    needed_signal_names,
    current=True,
    db_path=sqlite_db_path,
    chunksize=500_000,
)

combined_date_diag = candidate_signals.groupby('signal_name').agg(
    rows=('signal_name', 'size'),
    date_nulls=('Date', lambda x: x.isna().sum()),
    unique_dates=('Date', 'nunique'),
    unique_tickers=('ticker', 'nunique'),
    signal_value_nulls=('signal_value', lambda x: x.isna().sum()),
).sort_values('date_nulls', ascending=False)

display(combined_date_diag)
validate_signal_date_quality(candidate_signals, context='03D combined candidate_signals', max_null_rate=0.0)

display(
    pd.DataFrame(
        [
            {
                'rows': len(candidate_signals),
                'requested_signals': len(needed_signal_names),
                'loaded_signals': candidate_signals['signal_name'].nunique(),
                'n_tickers': candidate_signals['ticker'].nunique(),
                'start_date': candidate_signals['Date'].min(),
                'end_date': candidate_signals['Date'].max(),
            }
        ]
    )
)

needed_signal_names: ['close_position_reversal_5', 'dollar_volume_shock_20', 'expanded_beta_adjusted_residual_20', 'expanded_distance_ma_10', 'expanded_distance_ma_20', 'expanded_residual_market_return_20', 'expanded_reversal_1d', 'expanded_reversal_3d', 'expanded_reversal_5d', 'expanded_zscore_reversal_20', 'failed_breakout_reversal_20', 'intraday_reversal_strength_1', 'liquidity_adjusted_reversal_5', 'overnight_gap_reversal_1', 'price_impact_proxy_20', 'range_compression_breakout_10', 'range_expansion_failure_5', 'relative_return_rank_20', 'relative_return_zscore_60', 'residual_return_vs_universe_20', 'three_day_overextension_reversal', 'vol_of_vol_20', 'vol_surprise_20_60']
count: 23


,signal_name,rows,min_date,max_date,date_nulls,sample_dates
0,close_position_reversal_5,1002844,2018-01-02 00:00:00,2026-05-07 00:00:00,0,"[2018-01-02 00:00:00, 2018-01-02 00:00:00, 201..."
1,dollar_volume_shock_20,1002844,2018-01-02 00:00:00,2026-05-07 00:00:00,0,"[2018-01-02 00:00:00, 2018-01-02 00:00:00, 201..."
2,expanded_beta_adjusted_residual_20,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."
3,expanded_distance_ma_10,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."
4,expanded_distance_ma_20,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."
5,expanded_residual_market_return_20,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."
6,expanded_reversal_1d,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."
7,expanded_reversal_3d,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."
8,expanded_reversal_5d,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."
9,expanded_zscore_reversal_20,1002844,2018-01-02,2026-05-07,0,"[2018-01-02, 2018-01-02, 2018-01-02, 2018-01-0..."


load_candidate_signals_by_names chunk 1: signals=['close_position_reversal_5'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 2: signals=['close_position_reversal_5'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 3: signals=['close_position_reversal_5'], rows=2,844, Date nulls=0
load_candidate_signals_by_names: table=candidate_signals_current, requested_signal_names=1, rows_returned=1,002,844, elapsed_seconds=1.647, memory_before_mb=125.4, memory_after_mb=725.8
close_position_reversal_5 1002844 datetime64[ns] 0 2098
load_candidate_signals_by_names chunk 1: signals=['dollar_volume_shock_20'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 2: signals=['dollar_volume_shock_20'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 3: signals=['dollar_volume_shock_20'], rows=2,844, Date nulls=0
load_candidate_signals_by_names: table=candidate_signals_current, requested_signal_names=1, rows_returned=1,002,844, elapsed_seco

,signal_name,rows,Date_dtype,date_nulls,unique_dates
0,close_position_reversal_5,1002844,datetime64[ns],0,2098
12,liquidity_adjusted_reversal_5,1002844,datetime64[ns],0,2098
21,vol_of_vol_20,1002844,datetime64[ns],0,2098
20,three_day_overextension_reversal,1002844,datetime64[ns],0,2098
19,residual_return_vs_universe_20,1002844,datetime64[ns],0,2098
18,relative_return_zscore_60,1002844,datetime64[ns],0,2098
17,relative_return_rank_20,1002844,datetime64[ns],0,2098
16,range_expansion_failure_5,1002844,datetime64[ns],0,2098
15,range_compression_breakout_10,1002844,datetime64[ns],0,2098
14,price_impact_proxy_20,1002844,datetime64[ns],0,2098


load_candidate_signals_by_names chunk 1: signals=['close_position_reversal_5'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 2: signals=['close_position_reversal_5'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 3: signals=['close_position_reversal_5', 'dollar_volume_shock_20'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 4: signals=['dollar_volume_shock_20'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 5: signals=['dollar_volume_shock_20', 'expanded_beta_adjusted_residual_20'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 6: signals=['expanded_beta_adjusted_residual_20'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 7: signals=['expanded_beta_adjusted_residual_20', 'expanded_distance_ma_10'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 8: signals=['expanded_distance_ma_10'], rows=500,000, Date nulls=0
load_candidate_signals_by_names chunk 9: si

,rows,date_nulls,unique_dates,unique_tickers,signal_value_nulls
signal_name,,,,,
close_position_reversal_5,1002844,0,2098,478,193533
liquidity_adjusted_reversal_5,1002844,0,2098,478,132376
vol_of_vol_20,1002844,0,2098,478,119758
three_day_overextension_reversal,1002844,0,2098,478,112161
residual_return_vs_universe_20,1002844,0,2098,478,362420
relative_return_zscore_60,1002844,0,2098,478,324834
relative_return_rank_20,1002844,0,2098,478,123901
range_expansion_failure_5,1002844,0,2098,478,127863
range_compression_breakout_10,1002844,0,2098,478,194051


03D combined candidate_signals Date quality by signal_name:
                       signal_name    rows  date_nulls  unique_dates  unique_tickers  date_null_rate
         close_position_reversal_5 1002844           0          2098             478             0.0
            dollar_volume_shock_20 1002844           0          2098             478             0.0
expanded_beta_adjusted_residual_20 1002844           0          2098             478             0.0
           expanded_distance_ma_10 1002844           0          2098             478             0.0
           expanded_distance_ma_20 1002844           0          2098             478             0.0
expanded_residual_market_return_20 1002844           0          2098             478             0.0
              expanded_reversal_1d 1002844           0          2098             478             0.0
              expanded_reversal_3d 1002844           0          2098             478             0.0
              expanded_reversal

,rows,requested_signals,loaded_signals,n_tickers,start_date,end_date
0,23065412,23,23,478,2018-01-02,2026-05-07


## 5. Load signal_scores_current and signal_best_horizon_current

In [4]:
display(signal_scores.head())
display(signal_best_horizon.head())
display(pd.DataFrame([{'signal_score_pairs': len(signal_scores), 'best_horizon_rows': len(signal_best_horizon)}]))

,signal_name,horizon,method,n_obs,mean_ic,median_ic,ic_std,ic_ir,hit_rate,positive_ic_rate,missing_pct,signal_family,signal_version,run_id,scoring_version
0,close_position_reversal_5,1,spearman,600770,0.006244,0.000382,0.190790,0.032728,0.499689,0.500491,0.400934,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
1,close_position_reversal_5,5,spearman,589206,0.007745,0.003505,0.179986,0.043031,0.496022,0.508358,0.412465,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
2,close_position_reversal_5,10,spearman,578401,0.006744,0.008758,0.173888,0.038786,0.494546,0.517003,0.423239,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
3,close_position_reversal_5,20,spearman,562984,0.008850,0.009159,0.166689,0.053094,0.494270,0.526994,0.438613,microstructure_lite,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2
4,dollar_volume_shock_20,1,spearman,594678,0.002990,0.004823,0.092843,0.032205,0.500327,0.523360,0.407008,liquidity_flow,phase2_orthogonal_signals_v2,signal_scoring_20260510_222922,phase2_signal_scoring_v2


,signal_name,signal_family,best_horizon,best_mean_ic,best_abs_mean_ic,best_ic_ir,best_positive_ic_rate,best_hit_rate,signal_direction,signal_strength,run_id,scoring_version
0,expanded_reversal_1d,mean_reversion,1,0.017276,0.017276,0.084684,0.529553,0.505222,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
1,expanded_residual_market_return_20,residual_relative_value,20,0.016748,0.016748,0.088930,0.528145,0.501801,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
2,vol_of_vol_20,volatility_structure,20,0.015976,0.015976,0.128788,0.553403,0.501826,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
3,expanded_reversal_3d,mean_reversion,1,0.014939,0.014939,0.070354,0.521886,0.503985,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2
4,range_expansion_failure_5,volatility_structure,20,0.014333,0.014333,0.112310,0.543756,0.497760,POSITIVE_EDGE,WEAK,signal_scoring_20260510_222922,phase2_signal_scoring_v2


,signal_score_pairs,best_horizon_rows
0,92,23


## 6. Load clean close prices

In [5]:
close_prices = load_price_table('clean_close_prices_current', db_path=sqlite_db_path)

display(pd.DataFrame([{'rows': close_prices.shape[0], 'columns': close_prices.shape[1]}]))
display(close_prices.head())

,rows,columns
0,2098,478


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Build regime features

In [6]:
regime_features = build_regime_features_for_ic(close_prices, benchmark_ticker='SPY')

regime_feature_coverage = (
    regime_features[REGIME_COLUMNS]
    .notna()
    .mean()
    .rename('coverage_pct')
    .reset_index()
    .rename(columns={'index': 'regime_column'})
)

display(regime_features.head())
display(regime_feature_coverage)

,Date,benchmark_return_1d,benchmark_vol_20d,benchmark_vol_regime,benchmark_trend_regime,market_drawdown,drawdown_regime,correlation_20d,correlation_regime
0,2018-01-02,NaN,NaN,NaN,NaN,0.0,LOW_DRAWDOWN,NaN,NaN
1,2018-01-03,0.006325,NaN,NaN,NaN,0.0,LOW_DRAWDOWN,NaN,NaN
2,2018-01-04,0.004215,NaN,NaN,NaN,0.0,LOW_DRAWDOWN,NaN,NaN
3,2018-01-05,0.006664,NaN,NaN,NaN,0.0,LOW_DRAWDOWN,NaN,NaN
4,2018-01-08,0.001829,NaN,NaN,NaN,0.0,LOW_DRAWDOWN,NaN,NaN


,regime_column,coverage_pct
0,benchmark_vol_regime,0.990467
1,benchmark_trend_regime,0.905148
2,drawdown_regime,1.000000
3,correlation_regime,0.983317


## 8. Run regime-conditioned IC analysis

In [7]:
candidate_signals_long = candidate_signals.copy()

print(candidate_signals_long.shape)
print(candidate_signals_long.columns.tolist())
print(candidate_signals_long.dtypes)
display(candidate_signals_long.head())
display(candidate_signals_long[candidate_signals_long['signal_name'].eq('expanded_reversal_1d')].head(20))

date_diag = candidate_signals_long.groupby('signal_name').agg(
    rows=('signal_name', 'size'),
    date_nulls=('Date', lambda x: x.isna().sum()),
    unique_dates=('Date', 'nunique'),
    unique_tickers=('ticker', 'nunique'),
    signal_value_nulls=('signal_value', lambda x: x.isna().sum()),
).sort_values('date_nulls', ascending=False)

display(date_diag.head(30))

bad = candidate_signals_long[candidate_signals_long['signal_name'] == 'expanded_reversal_1d']
display(bad[['Date', 'ticker', 'signal_name', 'signal_value']].head(30))
print(bad['Date'].head(30).tolist())
print(bad['Date'].dtype)
print('Date nulls:', bad['Date'].isna().sum())
print('Rows:', len(bad))
print('Unique dates:', bad['Date'].nunique(dropna=True))
print('Unique tickers:', bad['ticker'].nunique(dropna=True))

required_signal_long_columns = ['signal_name', 'Date', 'ticker', 'signal_value']
missing_signal_long_columns = [
    column for column in required_signal_long_columns if column not in candidate_signals_long.columns
]
if missing_signal_long_columns:
    raise ValueError(f'candidate_signals_long missing required columns: {missing_signal_long_columns}')
if not pd.api.types.is_datetime64_any_dtype(candidate_signals_long['Date']):
    raise TypeError(f"candidate_signals_long Date must be datetime64, got {candidate_signals_long['Date'].dtype}")
if not pd.api.types.is_numeric_dtype(candidate_signals_long['signal_value']):
    raise TypeError(
        'candidate_signals_long signal_value must be numeric, '
        f"got {candidate_signals_long['signal_value'].dtype}"
    )

date_null_count = int(candidate_signals_long['Date'].isna().sum())
date_null_rate = candidate_signals_long['Date'].isna().mean() if len(candidate_signals_long) else 0.0
if date_null_count > 0:
    null_date_by_signal = (
        candidate_signals_long.loc[candidate_signals_long['Date'].isna()]
        .groupby('signal_name', dropna=False)
        .size()
        .sort_values(ascending=False)
    )
    display(null_date_by_signal)
    raise ValueError(
        'candidate_signals_long contains Date nulls and cannot proceed: '
        f'date_nulls={date_null_count:,}, date_null_rate={date_null_rate:.2%}'
    )

validate_signal_date_quality(
    candidate_signals_long,
    context='03D candidate_signals_long',
    max_null_rate=0.0,
)

validate_signal_long_uniqueness(
    candidate_signals_long,
    key_cols=['signal_name', 'Date', 'ticker'],
    strict=True,
    context='03D candidate_signals_long',
)

regime_features, daily_regime_ic, regime_summary, regime_fragility = run_regime_ic_analysis(
    candidate_signals_long=candidate_signals_long,
    close_prices=close_prices,
    horizons=HORIZONS,
    regime_columns=REGIME_COLUMNS,
    signal_scores=signal_scores,
    method=IC_METHOD,
)

display(
    pd.DataFrame(
        [
            {
                'regime_features_rows': len(regime_features),
                'daily_regime_ic_rows': len(daily_regime_ic),
                'regime_summary_rows': len(regime_summary),
                'regime_fragility_rows': len(regime_fragility),
            }
        ]
    )
)
display(daily_regime_ic.head())

(23065412, 6)
['Date', 'ticker', 'signal_name', 'signal_value', 'run_id', 'signal_version']
Date              datetime64[ns]
ticker                    object
signal_name               object
signal_value             float64
run_id                    object
signal_version            object
dtype: object


,Date,ticker,signal_name,signal_value,run_id,signal_version
0,2018-01-02,A,close_position_reversal_5,NaN,phase2_orthogonal_signals_20260508_075857,None
1,2018-01-02,AAPL,close_position_reversal_5,NaN,phase2_orthogonal_signals_20260508_075857,None
2,2018-01-02,ABBV,close_position_reversal_5,NaN,phase2_orthogonal_signals_20260508_075857,None
3,2018-01-02,ABNB,close_position_reversal_5,NaN,phase2_orthogonal_signals_20260508_075857,None
4,2018-01-02,ABT,close_position_reversal_5,NaN,phase2_orthogonal_signals_20260508_075857,None


,Date,ticker,signal_name,signal_value,run_id,signal_version
6017064,2018-01-02,A,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017065,2018-01-02,AAPL,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017066,2018-01-02,ABBV,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017067,2018-01-02,ABNB,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017068,2018-01-02,ABT,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017069,2018-01-02,ACGL,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017070,2018-01-02,ACN,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017071,2018-01-02,ADBE,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017072,2018-01-02,ADI,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1
6017073,2018-01-02,ADM,expanded_reversal_1d,NaN,phase2_expanded_discovery_20260509_235638,phase2_expanded_discovery_v1


,rows,date_nulls,unique_dates,unique_tickers,signal_value_nulls
signal_name,,,,,
close_position_reversal_5,1002844,0,2098,478,193533
liquidity_adjusted_reversal_5,1002844,0,2098,478,132376
vol_of_vol_20,1002844,0,2098,478,119758
three_day_overextension_reversal,1002844,0,2098,478,112161
residual_return_vs_universe_20,1002844,0,2098,478,362420
relative_return_zscore_60,1002844,0,2098,478,324834
relative_return_rank_20,1002844,0,2098,478,123901
range_expansion_failure_5,1002844,0,2098,478,127863
range_compression_breakout_10,1002844,0,2098,478,194051


,Date,ticker,signal_name,signal_value
6017064,2018-01-02,A,expanded_reversal_1d,NaN
6017065,2018-01-02,AAPL,expanded_reversal_1d,NaN
6017066,2018-01-02,ABBV,expanded_reversal_1d,NaN
6017067,2018-01-02,ABNB,expanded_reversal_1d,NaN
6017068,2018-01-02,ABT,expanded_reversal_1d,NaN
6017069,2018-01-02,ACGL,expanded_reversal_1d,NaN
6017070,2018-01-02,ACN,expanded_reversal_1d,NaN
6017071,2018-01-02,ADBE,expanded_reversal_1d,NaN
6017072,2018-01-02,ADI,expanded_reversal_1d,NaN
6017073,2018-01-02,ADM,expanded_reversal_1d,NaN


[Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('2018-01-02 00:00:00'), Timestamp('20

,regime_features_rows,daily_regime_ic_rows,regime_summary_rows,regime_fragility_rows
0,2098,772064,1012,368


,Date,regime_column,regime_value,daily_ic,signal_name,horizon,method
0,2018-01-02,benchmark_vol_regime,NaN,NaN,close_position_reversal_5,1,spearman
1,2018-01-03,benchmark_vol_regime,NaN,NaN,close_position_reversal_5,1,spearman
2,2018-01-04,benchmark_vol_regime,NaN,NaN,close_position_reversal_5,1,spearman
3,2018-01-05,benchmark_vol_regime,NaN,NaN,close_position_reversal_5,1,spearman
4,2018-01-08,benchmark_vol_regime,NaN,NaN,close_position_reversal_5,1,spearman


## 9. Build regime summaries and fragility report

In [8]:
regime_summary_enriched = regime_summary.merge(
    signal_best_horizon[['signal_name', 'signal_family', 'best_horizon', 'signal_direction', 'signal_strength']],
    on='signal_name',
    how='left',
)
regime_fragility_enriched = regime_fragility.merge(
    signal_best_horizon[['signal_name', 'signal_family', 'best_horizon', 'signal_direction', 'signal_strength']],
    on='signal_name',
    how='left',
)
regime_opportunity_summary = build_regime_opportunity_summary(
    regime_summary=regime_summary_enriched,
    fragility=regime_fragility_enriched,
)

opportunity_display_columns = [
    'signal_name',
    'horizon',
    'signal_family',
    'best_regime_column',
    'best_regime_value',
    'best_abs_mean_ic',
    'regime_sample_weight',
    'adjusted_best_abs_ic',
    'regime_consistency_score',
    'regime_fragility_flag',
    'recommended_use',
    'opportunity_notes',
]

top_regime_conditioned = (
    regime_summary_enriched.sort_values(['abs_mean_ic', 'n_obs'], ascending=[False, False])
    .head(20)
)
most_regime_fragile = (
    regime_fragility_enriched.sort_values(
        ['regime_fragility_flag', 'regime_dependency_ratio', 'regime_ic_spread'],
        ascending=[True, False, False],
    )
    .head(20)
)
lowest_fragility = (
    regime_fragility_enriched.loc[
        regime_fragility_enriched['regime_fragility_flag'].eq('LOW_REGIME_FRAGILITY')
    ]
    .sort_values(['regime_dependency_ratio', 'regime_ic_spread'])
    .head(20)
)
recommended_use_counts = regime_opportunity_summary['recommended_use'].value_counts()
fragility_counts = regime_fragility_enriched['regime_fragility_flag'].value_counts()
top_conditional_opportunities = (
    regime_opportunity_summary.loc[regime_opportunity_summary['recommended_use'].eq('CONDITIONAL')]
    .sort_values(['adjusted_best_abs_ic', 'best_abs_mean_ic'], ascending=[False, False])
    .head(20)
)
top_global_opportunities = (
    regime_opportunity_summary.loc[regime_opportunity_summary['recommended_use'].eq('GLOBAL')]
    .sort_values(['adjusted_best_abs_ic', 'best_abs_mean_ic'], ascending=[False, False])
    .head(20)
)
avoid_examples = (
    regime_opportunity_summary.loc[regime_opportunity_summary['recommended_use'].eq('AVOID')]
    .sort_values(['adjusted_best_abs_ic', 'best_abs_mean_ic'])
    .head(20)
)

print('Regime summary sample')
display(regime_summary_enriched.head())

print('Regime fragility sample')
display(regime_fragility_enriched.head())

print('Regime opportunity summary sample')
display(regime_opportunity_summary[opportunity_display_columns].head())


Regime summary sample


,signal_name,horizon,regime_column,regime_value,n_obs,mean_ic,median_ic,ic_std,ic_ir,positive_ic_rate,abs_mean_ic,signal_family,best_horizon,signal_direction,signal_strength
0,close_position_reversal_5,1,benchmark_trend_regime,DOWNTREND,281,0.004265,-0.004701,0.188349,0.022643,0.487544,0.004265,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
1,close_position_reversal_5,1,benchmark_trend_regime,SIDEWAYS,240,0.027238,0.015899,0.248213,0.109736,0.512500,0.027238,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
2,close_position_reversal_5,1,benchmark_trend_regime,UPTREND,1377,0.002320,0.001685,0.183570,0.012640,0.501816,0.002320,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
3,close_position_reversal_5,1,benchmark_vol_regime,HIGH_VOL,666,0.013249,0.008282,0.222460,0.059556,0.512012,0.013249,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
4,close_position_reversal_5,1,benchmark_vol_regime,LOW_VOL,690,-0.000273,-0.008897,0.164326,-0.001664,0.479710,0.000273,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL


Regime fragility sample


,signal_name,horizon,regime_column,best_regime,worst_regime,best_abs_mean_ic,worst_abs_mean_ic,regime_ic_spread,regime_dependency_ratio,sign_flip_across_regimes,regime_fragility_flag,signal_family,best_horizon,signal_direction,signal_strength
0,close_position_reversal_5,1,benchmark_trend_regime,SIDEWAYS,UPTREND,0.027238,0.002320,0.024918,11.739255,False,HIGH_REGIME_FRAGILITY,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
1,close_position_reversal_5,1,benchmark_vol_regime,HIGH_VOL,LOW_VOL,0.013249,0.000273,0.012975,48.465290,True,HIGH_REGIME_FRAGILITY,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
2,close_position_reversal_5,1,correlation_regime,MID_CORR,LOW_CORR,0.010116,0.000073,0.010042,137.662359,False,HIGH_REGIME_FRAGILITY,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
3,close_position_reversal_5,1,drawdown_regime,HIGH_DRAWDOWN,LOW_DRAWDOWN,0.008699,0.005667,0.003032,1.534994,False,LOW_REGIME_FRAGILITY,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL
4,close_position_reversal_5,5,benchmark_trend_regime,SIDEWAYS,UPTREND,0.051556,0.003451,0.048105,14.937843,True,HIGH_REGIME_FRAGILITY,microstructure_lite,20,POSITIVE_EDGE,NO_SIGNAL


Regime opportunity summary sample


,signal_name,horizon,signal_family,best_regime_column,best_regime_value,best_abs_mean_ic,regime_sample_weight,adjusted_best_abs_ic,regime_consistency_score,regime_fragility_flag,recommended_use,opportunity_notes
0,close_position_reversal_5,1,microstructure_lite,benchmark_trend_regime,SIDEWAYS,0.027238,0.8,0.021790,1.000000,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...
1,close_position_reversal_5,5,microstructure_lite,benchmark_trend_regime,SIDEWAYS,0.051556,0.8,0.041245,0.666667,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...
2,close_position_reversal_5,10,microstructure_lite,benchmark_trend_regime,SIDEWAYS,0.049799,0.8,0.039839,0.333333,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; High dependenc...
3,close_position_reversal_5,20,microstructure_lite,correlation_regime,HIGH_CORR,0.038474,1.0,0.038474,0.333333,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; low regime sig...
4,dollar_volume_shock_20,1,liquidity_flow,correlation_regime,MID_CORR,0.011396,1.0,0.011396,0.666667,HIGH_REGIME_FRAGILITY,AVOID,Weak across regimes; High dependency ratio due...


## 10. Save outputs to SQLite

In [9]:
saved_paths = save_regime_ic_outputs(
    regime_features=regime_features,
    daily_regime_ic=daily_regime_ic,
    regime_summary=regime_summary_enriched,
    regime_fragility=regime_fragility_enriched,
    regime_opportunity_summary=regime_opportunity_summary,
    db_path=sqlite_db_path,
    run_id=run_id,
    regime_ic_version=REGIME_IC_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            'artifact': artifact,
            'current_table': tables[0],
            'history_table': tables[1],
            'sqlite_path': str(saved_paths[artifact]),
        }
        for artifact, tables in REGIME_IC_TABLES.items()
    ]
)

display(sqlite_tables_written)


,artifact,current_table,history_table,sqlite_path
0,features,regime_features_ic_current,regime_features_ic_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,daily,signal_regime_ic_daily_current,signal_regime_ic_daily_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,summary,signal_regime_ic_summary_current,signal_regime_ic_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,fragility,signal_regime_fragility_current,signal_regime_fragility_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,opportunity,signal_regime_opportunity_summary_current,signal_regime_opportunity_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 11. Final display

In [10]:
print('Regime feature coverage')
display(regime_feature_coverage)

print('Regime fragility counts')
display(fragility_counts.rename('signal_horizon_regime_count'))

print('Recommended use counts')
display(recommended_use_counts.rename('signal_horizon_count'))

print('Top CONDITIONAL opportunities by adjusted_best_abs_ic')
display(top_conditional_opportunities[opportunity_display_columns])

print('Top GLOBAL opportunities by adjusted_best_abs_ic')
display(top_global_opportunities[opportunity_display_columns])

print('AVOID examples')
display(avoid_examples[opportunity_display_columns])

print('SQLite tables written')
display(sqlite_tables_written)


Regime feature coverage


,regime_column,coverage_pct
0,benchmark_vol_regime,0.990467
1,benchmark_trend_regime,0.905148
2,drawdown_regime,1.000000
3,correlation_regime,0.983317


Regime fragility counts


regime_fragility_flag
HIGH_REGIME_FRAGILITY        279
MODERATE_REGIME_FRAGILITY     70
LOW_REGIME_FRAGILITY          19
Name: signal_horizon_regime_count, dtype: int64

Recommended use counts


...

,signal_name,horizon,signal_family,best_regime_column,best_regime_value,best_abs_mean_ic,regime_sample_weight,adjusted_best_abs_ic,regime_consistency_score,regime_fragility_flag,recommended_use,opportunity_notes


AVOID examples


,signal_name,horizon,signal_family,best_regime_column,best_regime_value,best_abs_mean_ic,regime_sample_weight,adjusted_best_abs_ic,regime_consistency_score,regime_fragility_flag,recommended_use,opportunity_notes
88,vol_surprise_20_60,1,volatility_structure,benchmark_vol_regime,MID_VOL,0.007664,1.000000,0.007664,0.333333,HIGH_REGIME_FRAGILITY,AVOID,Weak across regimes; low regime sign consistency
7,dollar_volume_shock_20,20,liquidity_flow,benchmark_trend_regime,SIDEWAYS,0.009766,0.800000,0.007813,0.666667,HIGH_REGIME_FRAGILITY,AVOID,Weak across regimes; sample-size adjusted
52,overnight_gap_reversal_1,1,true_short_term_reversal,benchmark_trend_regime,DOWNTREND,0.010609,0.936667,0.009937,1.000000,MODERATE_REGIME_FRAGILITY,AVOID,Weak across regimes; sample-size adjusted; hig...
4,dollar_volume_shock_20,1,liquidity_flow,correlation_regime,MID_CORR,0.011396,1.000000,0.011396,0.666667,HIGH_REGIME_FRAGILITY,AVOID,Weak across regimes; High dependency ratio due...


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,features,regime_features_ic_current,regime_features_ic_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,daily,signal_regime_ic_daily_current,signal_regime_ic_daily_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,summary,signal_regime_ic_summary_current,signal_regime_ic_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,fragility,signal_regime_fragility_current,signal_regime_fragility_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,opportunity,signal_regime_opportunity_summary_current,signal_regime_opportunity_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [11]:
from src.db import load_table

opp = load_table("signal_regime_opportunity_summary_current")

print(opp.shape)
print(opp[["regime_sample_weight", "adjusted_best_abs_ic", "regime_consistency_score"]].isna().sum())

display(opp["recommended_use"].value_counts())

display(
    opp[opp["recommended_use"] == "CONDITIONAL"]
    .sort_values("adjusted_best_abs_ic", ascending=False)
    .head(20)
)

(92, 25)
regime_sample_weight        0
adjusted_best_abs_ic        0
regime_consistency_score    0
dtype: int64


recommended_use
CONDITIONAL    74
WATCHLIST      14
AVOID           4
Name: count, dtype: int64

,signal_name,horizon,signal_family,signal_direction,signal_strength,best_regime_column,best_regime_value,best_abs_mean_ic,regime_sample_weight,adjusted_best_abs_ic,...,worst_regime_value,worst_abs_mean_ic,regime_ic_spread,regime_dependency_ratio,sign_flip_across_regimes,regime_fragility_flag,recommended_use,opportunity_notes,run_id,regime_ic_version
87,vol_of_vol_20,20,volatility_structure,POSITIVE_EDGE,WEAK,benchmark_trend_regime,DOWNTREND,0.085258,0.936667,0.079858,...,UPTREND,0.010232,0.075026,8.332596,1,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
23,expanded_residual_market_return_20,20,residual_relative_value,POSITIVE_EDGE,WEAK,benchmark_trend_regime,DOWNTREND,0.076801,0.936667,0.071937,...,SIDEWAYS,0.000922,0.075879,83.288756,0,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; High dependenc...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
59,price_impact_proxy_20,20,liquidity_flow,POSITIVE_EDGE,NO_SIGNAL,benchmark_trend_regime,DOWNTREND,0.076435,0.936667,0.071594,...,UPTREND,0.006239,0.070196,12.251350,1,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
86,vol_of_vol_20,10,volatility_structure,POSITIVE_EDGE,WEAK,benchmark_trend_regime,DOWNTREND,0.066914,0.936667,0.062676,...,UPTREND,0.009066,0.057848,7.380584,1,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
43,failed_breakout_reversal_20,20,microstructure_lite,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK,drawdown_regime,HIGH_DRAWDOWN,0.055482,1.000000,0.055482,...,LOW_DRAWDOWN,0.003819,0.051663,14.527756,0,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; high regime si...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
19,expanded_distance_ma_20,20,mean_reversion,POSITIVE_EDGE,WEAK,drawdown_regime,HIGH_DRAWDOWN,0.054731,1.000000,0.054731,...,LOW_DRAWDOWN,0.002590,0.052141,21.135427,0,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; high regime si...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
58,price_impact_proxy_20,10,liquidity_flow,POSITIVE_EDGE,NO_SIGNAL,benchmark_trend_regime,DOWNTREND,0.057810,0.936667,0.054149,...,UPTREND,0.006633,0.051177,8.715533,1,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
63,range_compression_breakout_10,20,microstructure_lite,NEGATIVE_EDGE_REVERSE_SIGNAL,WEAK,correlation_regime,HIGH_CORR,0.053203,1.000000,0.053203,...,MID_CORR,0.001952,0.051251,27.249743,1,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; low regime sig...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
75,relative_return_zscore_60,20,cross_sectional_relative_value,NEGATIVE_EDGE_REVERSE_SIGNAL,NO_SIGNAL,benchmark_trend_regime,DOWNTREND,0.055075,0.936667,0.051587,...,SIDEWAYS,0.005466,0.049610,10.076799,1,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
11,expanded_beta_adjusted_residual_20,20,residual_relative_value,POSITIVE_EDGE,NO_SIGNAL,benchmark_trend_regime,DOWNTREND,0.053589,0.936667,0.050195,...,UPTREND,0.004976,0.048613,10.768898,1,HIGH_REGIME_FRAGILITY,CONDITIONAL,Strong conditional regime edge; sample-size ad...,phase2_nb03d_regime_ic_20260510_215329,phase2_regime_ic_v1
